In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

In [2]:
# 1. Dim_Regions
regions_data = [
    {"RegionID": 1, "RegionName": "North Hub", "Territory": "Metro"},
    {"RegionID": 2, "RegionName": "South Hub", "Territory": "Coastal"},
    {"RegionID": 3, "RegionName": "East Hub", "Territory": "Regional"},
    {"RegionID": 4, "RegionName": "West Hub", "Territory": "Inland"}
]
df_regions = pd.DataFrame(regions_data)
df_regions.to_csv("dim_regions.csv", index=False)

In [3]:
# 2. Dim_Technicians
techs_data = [
    {"TechnicianID": 101, "TechnicianName": "Ramesh Kumar", "VendorName": "Apex Infra"},
    {"TechnicianID": 102, "TechnicianName": "Priya Sharma", "VendorName": "Apex Infra"},
    {"TechnicianID": 103, "TechnicianName": "David Miller", "VendorName": "Delta Services"},
    {"TechnicianID": 104, "TechnicianName": "Anita Rao", "VendorName": "Delta Services"},
    {"TechnicianID": 105, "TechnicianName": "Vikram Singh", "VendorName": "Zenith Connect"}
]
df_techs = pd.DataFrame(techs_data)
df_techs.to_csv("dim_technicians.csv", index=False)

In [4]:
# 3. Fact_Audits
start_date = datetime(2025, 10, 1)
statuses = ['PASS', 'PASSED', 'FAIL', 'FAILED', 'REJECTED', 'PENDING']
defects = ['Cable Sagging', 'Loose Connector', 'Grounding Missing', 'Signal Attenuation']

audits = []
for i in range(1, 601):
    sched_date = start_date + timedelta(days=random.randint(0, 300))
    status = random.choices(statuses, weights=[0.45, 0.20, 0.15, 0.08, 0.07, 0.05])[0]
    
    # Realistic business logic
    if status == 'PENDING':
        audit_date = None
        defect = None
    else:
        delay = random.choice([0, 1, 2, 3, 5, 7]) # execution turnaround
        audit_date = sched_date + timedelta(days=delay)
        defect = random.choice(defects) if status in ['FAIL', 'FAILED', 'REJECTED'] else None
    
    audits.append({
        "AuditID": 1000 + i,
        "SiteID": f"SITE-{random.randint(201, 240)}",
        "RegionID": random.choice([1, 2, 3, 4]),
        "TechnicianID": random.choice([101, 102, 103, 104, 105]),
        "ScheduledDate": sched_date.strftime('%Y-%m-%d'),
        "AuditDate": audit_date.strftime('%Y-%m-%d') if audit_date else None,
        "StatusCode": status,
        "DefectCategory": defect,
        "IsTestRecord": 1 if i in [12, 45, 88] else 0
    })

df_audits = pd.DataFrame(audits)
df_audits.to_csv("fact_audits.csv", index=False)

In [5]:
# --- Exploratory Data Analysis & Quality Profiling ---
print("=== FACT AUDITS PROFILE ===")
print(f"Total Ingested Rows: {len(df_audits)}")
print(f"Test Records Identified: {(df_audits['IsTestRecord'] == 1).sum()}")

print("\n--- Raw Status Distribution ---")
print(df_audits['StatusCode'].value_counts(dropna=False))

print("\n--- Defect Distribution ---")
print(df_audits['DefectCategory'].value_counts(dropna=False))

print("\n--- Date Boundary Verification ---")
sched_dt = pd.to_datetime(df_audits['ScheduledDate'])
audit_dt = pd.to_datetime(df_audits['AuditDate'])

print(f"Scheduled: {sched_dt.min().strftime('%Y-%m-%d')} to {sched_dt.max().strftime('%Y-%m-%d')}")
print(f"Executed:  {audit_dt.min().strftime('%Y-%m-%d')} to {audit_dt.max().strftime('%Y-%m-%d')}")
print(f"Pending Audits (Null Execution): {df_audits['AuditDate'].isna().sum()}")

=== FACT AUDITS PROFILE ===
Total Ingested Rows: 600
Test Records Identified: 3

--- Raw Status Distribution ---
StatusCode
PASS        279
PASSED      129
FAIL         88
FAILED       40
REJECTED     33
PENDING      31
Name: count, dtype: int64

--- Defect Distribution ---
DefectCategory
None                  439
Loose Connector        54
Signal Attenuation     44
Grounding Missing      36
Cable Sagging          27
Name: count, dtype: int64

--- Date Boundary Verification ---
Scheduled: 2025-10-02 to 2026-07-28
Executed:  2025-10-04 to 2026-08-01
Pending Audits (Null Execution): 31


In [6]:
# --- Gate 1: Pre-Ingestion Schema & Business Rule Assertions (ANZSCO 263211 & 261111) ---
print("=== EXECUTING GATE 1: PRE-INGESTION QUALITY ASSERTIONS ===")

# 1. Primary Key Uniqueness & Non-Null Constraint (TC-01)
assert df_audits['AuditID'].is_unique, "Assertion Failure: Duplicate AuditID detected!"
assert df_audits['AuditID'].notna().all(), "Assertion Failure: Null AuditID detected!"
print("[PASS] Gate 1.1: AuditID primary key uniqueness and non-null constraint verified.")

# 2. Business Rule BR-02 / Status-Date Dependency Integrity (TC-02)
pending_violations = df_audits[(df_audits['StatusCode'] == 'PENDING') & (df_audits['AuditDate'].notna())]
assert len(pending_violations) == 0, f"Assertion Failure: {len(pending_violations)} pending audits have execution dates!"
print(f"[PASS] Gate 1.2: Status-Date dependency verified (31/31 PENDING audits enforce NULL execution date).")

# 3. Business Rule BR-01 / Turnaround Latency Boundary Gating (TC-03)
completed_audits = df_audits[df_audits['AuditDate'].notna()].copy()
latencies = (pd.to_datetime(completed_audits['AuditDate']) - pd.to_datetime(completed_audits['ScheduledDate'])).dt.days
assert (latencies >= 0).all(), "Assertion Failure: Negative turnaround latency detected!"
assert (latencies <= 7).all(), "Assertion Failure: Turnaround latency exceeded maximum 7-day boundary!"
print(f"[PASS] Gate 1.3: Turnaround latency bounded within operational limits (Min: {latencies.min()}d, Max: {latencies.max()}d).")

# 4. Referential Foreign Key Integrity (TC-04 & TC-05)
valid_regions = set(df_regions['RegionID'])
valid_techs = set(df_techs['TechnicianID'])
assert df_audits['RegionID'].isin(valid_regions).all(), "Assertion Failure: Orphan RegionID detected!"
assert df_audits['TechnicianID'].isin(valid_techs).all(), "Assertion Failure: Orphan TechnicianID detected!"
print("[PASS] Gate 1.4: Referential integrity verified against dim_regions and dim_technicians (0 orphans detected).")

# 5. Defect Attribution Completeness (TC-07)
failed_audits = df_audits[df_audits['StatusCode'].isin(['FAIL', 'FAILED', 'REJECTED'])]
assert failed_audits['DefectCategory'].notna().all(), "Assertion Failure: Unattributed defects found in failed audits!"
print(f"[PASS] Gate 1.5: Defect attribution complete (100% of {len(failed_audits)} non-compliant audits categorized).")

# 6. Test Record Quarantine Flagging (TC-06)
test_records = df_audits[df_audits['IsTestRecord'] == 1]
assert len(test_records) == 3, f"Assertion Failure: Expected 3 test records, found {len(test_records)}!"
print(f"[PASS] Gate 1.6: Automated test records quarantined for downstream filtering ({len(test_records)} records flagged).")

print("\n>>> ALL PRE-INGESTION QUALITY GATES PASSED (100% VERIFIED).")

=== EXECUTING GATE 1: PRE-INGESTION QUALITY ASSERTIONS ===
[PASS] Gate 1.1: AuditID primary key uniqueness and non-null constraint verified.
[PASS] Gate 1.2: Status-Date dependency verified (31/31 PENDING audits enforce NULL execution date).
[PASS] Gate 1.3: Turnaround latency bounded within operational limits (Min: 0d, Max: 7d).
[PASS] Gate 1.4: Referential integrity verified against dim_regions and dim_technicians (0 orphans detected).
[PASS] Gate 1.5: Defect attribution complete (100% of 161 non-compliant audits categorized).
[PASS] Gate 1.6: Automated test records quarantined for downstream filtering (3 records flagged).

>>> ALL PRE-INGESTION QUALITY GATES PASSED (100% VERIFIED).
